In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
conn = sqlite3.connect('../db.sqlite3')
query = """
SELECT 
    o.id AS order_id,
    o.datetime,
    oi.quantity,
    p.name AS product_name,
    p.price
FROM restaurant_order o
JOIN restaurant_orderitem oi ON o.id = oi.order_id
JOIN restaurant_product p ON oi.product_id = p.id
"""

df = pd.read_sql_query(query, conn)
df.to_csv("result.csv", index=False)
restaraunt = pd.read_csv('result.csv')
count_rows=len(restaraunt)
print(count_rows)

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
# write your code here
top_10_sales = (df.groupby('product_name')['quantity'].sum().sort_values(ascending=False).head(10)
)
plt.pie(top_10_sales, labels=top_10_sales.index, autopct='%1.1f%%', startangle=90
)

plt.title("Top 10 Most Popular Products (by Quantity)")
plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
# write your code here
df["item_price"] = df["price"] * df["quantity"]
top_10_revenue = (df.groupby("product_name")["item_price"].sum().sort_values(ascending=False).head(10)
)
plt.pie(
    top_10_revenue,
    labels=top_10_revenue.index,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Top 10 Products by Revenue")
plt.show()


# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
# write your code here
df["datetime"] = pd.to_datetime(df["datetime"])
df["order_hour"] = df["datetime"].dt.hour
revenue_by_hour = (
    df.groupby("order_hour")["item_price"]
    .sum()
    .sort_index()
)
plt.bar(revenue_by_hour.index, revenue_by_hour.values)
plt.xlabel("Hour of Day")
plt.ylabel("Total Revenue")
plt.title("Revenue by Order Hour")
plt.xticks(range(0, 24))  
plt.show()

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
# write your code here
df["datetime"] = pd.to_datetime(df["datetime"])
df["day_of_week"] = df["datetime"].dt.day_name()
days_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]
df["day_of_week"] = pd.Categorical(df["day_of_week"], categories=days_order, ordered=True)
revenue_by_day = (
    df.groupby("day_of_week", observed=False)["item_price"]
    .sum()
)
plt.bar(revenue_by_day.index, revenue_by_day.values)
plt.xlabel("Day of Week")
plt.ylabel("Total Revenue")
plt.title("Revenue by Day of Week")
plt.xticks(rotation=45)
plt.show()